In [ ]:
!pip -q install -U ydata-profiling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from ydata_profiling import ProfileReport

# Create a different dataset: Used Car Price Prediction
np.random.seed(42)
n = 300
df = pd.DataFrame({
    "Brand": np.random.choice(["Toyota", "Honda", "Hyundai", "Ford", "BMW"], n),
    "Fuel_Type": np.random.choice(["Petrol", "Diesel", "CNG", "Electric"], n),
    "Year": np.random.randint(2012, 2025, n),
    "Kilometers_Driven": np.random.randint(5000, 150000, n),
    "Engine_CC": np.random.randint(800, 3000, n),
    "Owner_Count": np.random.randint(1, 4, n),
    "Mileage": np.round(np.random.uniform(10, 30, n), 1)
})
brand_value = df["Brand"].map({"Toyota": 120000, "Honda": 110000, "Hyundai": 90000, "Ford": 85000, "BMW": 250000})
fuel_value = df["Fuel_Type"].map({"Petrol": 0, "Diesel": 30000, "CNG": 15000, "Electric": 100000})
df["Price"] = (brand_value + fuel_value + (df["Year"] - 2010) * 45000 - df["Kilometers_Driven"] * 0.7 + df["Engine_CC"] * 35 - df["Owner_Count"] * 12000 + df["Mileage"] * 1500 + np.random.normal(0, 18000, n)).round(0).clip(lower=30000)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nStatistics:\n", df.describe())

report = ProfileReport(df, title="Used Car Price Prediction EDA")
report.to_file("Car_Price_EDA_Report.html")

plt.figure(figsize=(8,5))
plt.hist(df["Price"], bins=20)
plt.xlabel("Car Price")
plt.ylabel("Frequency")
plt.title("Car Price Distribution")
plt.show()

df.groupby("Brand")["Price"].mean().sort_values().plot(kind="bar", figsize=(10,5))
plt.xlabel("Brand")
plt.ylabel("Average Price")
plt.title("Average Price by Brand")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["Kilometers_Driven"], df["Price"])
plt.xlabel("Kilometers Driven")
plt.ylabel("Price")
plt.title("Kilometers Driven vs Price")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["Engine_CC"], df["Price"])
plt.xlabel("Engine CC")
plt.ylabel("Price")
plt.title("Engine Size vs Price")
plt.show()

df = df.dropna(subset=["Price"])
X = df.drop("Price", axis=1)
y = df["Price"]

for column in X.select_dtypes(include="object").columns:
    X[column] = LabelEncoder().fit_transform(X[column].astype(str))
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)

bagging = BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=100, bootstrap=True, random_state=42)
bagging.fit(X_train, y_train)
bag_pred = bagging.predict(X_test)

rf = RandomForestRegressor(n_estimators=100, max_features="sqrt", bootstrap=True, oob_score=True, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))
bag_rmse = np.sqrt(mean_squared_error(y_test, bag_pred))
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
tree_mae = mean_absolute_error(y_test, tree_pred)
bag_mae = mean_absolute_error(y_test, bag_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
tree_r2 = r2_score(y_test, tree_pred)
bag_r2 = r2_score(y_test, bag_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("\nDecision Tree")
print("RMSE:", tree_rmse)
print("MAE:", tree_mae)
print("R2:", tree_r2)
print("\nBagging")
print("RMSE:", bag_rmse)
print("MAE:", bag_mae)
print("R2:", bag_r2)
print("\nRandom Forest")
print("RMSE:", rf_rmse)
print("MAE:", rf_mae)
print("R2:", rf_r2)
print("OOB Score:", rf.oob_score_)
print("OOB Error:", 1 - rf.oob_score_)

sample = X_train.sample(len(X_train), replace=True, random_state=42)
print("\nBootstrap Sampling")
print("Original Samples:", len(X_train))
print("Bootstrap Samples:", len(sample))
print("Unique Samples:", sample.index.nunique())
print("OOB Samples:", len(X_train) - sample.index.nunique())

print("\nFeature Subsampling")
for value in ["sqrt", "log2", 0.5, 1.0]:
    model = RandomForestRegressor(n_estimators=100, max_features=value, bootstrap=True, oob_score=True, random_state=42)
    model.fit(X_train, y_train)
    print("max_features =", value, "| OOB Score =", round(model.oob_score_, 4))

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importance:\n", importance)
importance.plot(kind="bar", figsize=(10,5))
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("Random Forest Feature Importance")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(y_test, rf_pred)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Car Price")
plt.show()

results = pd.DataFrame({"Model": ["Decision Tree", "Bagging", "Random Forest"], "RMSE": [tree_rmse, bag_rmse, rf_rmse], "MAE": [tree_mae, bag_mae, rf_mae], "R2": [tree_r2, bag_r2, rf_r2]})
print("\nModel Comparison:\n", results)
results.plot(x="Model", y="RMSE", kind="bar", figsize=(8,5))
plt.ylabel("RMSE")
plt.title("Model RMSE Comparison")
plt.show()

results.to_csv("Car_Price_Model_Results.csv", index=False)
importance.to_csv("Car_Price_Feature_Importance.csv")
print("\nFiles created: Car_Price_EDA_Report.html, Car_Price_Model_Results.csv, Car_Price_Feature_Importance.csv")
